installation + chargement modèle

In [ ]:
!pip install -q gradio transformers torch

from google.colab import drive
drive.mount('/content/drive')

import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE  = '/content/drive/MyDrive/FakeNewsProject'
MODEL = f'{BASE}/models/bert_final'

print("Chargement du modèle BERT...")
tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL)
mdl.eval()
print("✓ Modèle chargé")

Mounted at /content/drive
Chargement du modèle BERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Modèle chargé


interface Gradio complète

In [ ]:
def predict(title, text):
    if not text.strip():
        return "Veuillez entrer un texte", {"FAKE":0.0, "REAL":0.0}

    full = (title + " " + text)[:512]
    enc  = tok(full, return_tensors="pt", truncation=True, max_length=512)

    with torch.no_grad():
        probs = torch.softmax(mdl(**enc).logits, dim=-1)[0].tolist()

    fake_p, real_p = round(probs[0], 3), round(probs[1], 3)
    conf   = max(fake_p, real_p)
    label  = "🔴  FAKE NEWS" if fake_p > real_p else "🟢  REAL NEWS"
    verdict = f"{label}\nConfiance : {conf:.1%}\nFake : {fake_p:.1%}  |  Real : {real_p:.1%}"

    return verdict, {"FAKE": fake_p, "REAL": real_p}

examples = [
    ["BREAKING: Massive election fraud discovered",
     "Officials confirmed tonight that millions of votes were illegally changed in a conspiracy involving top government officials..."],
    ["Federal Reserve raises interest rates",
     "The Federal Reserve raised its benchmark interest rate by a quarter point on Wednesday, the tenth increase since March 2022..."],
    ["Scientists discover cure for all diseases",
     "A miraculous pill developed in secret labs will end all human suffering according to anonymous sources close to the matter..."]
]

demo = gr.Interface(
    fn          = predict,
    inputs      = [
        gr.Textbox(label="Titre de l'article",
                   placeholder="Ex: Scientists discover new vaccine..."),
        gr.Textbox(label="Texte de l'article", lines=8,
                   placeholder="Collez le contenu de l'article ici...")
    ],
    outputs     = [
        gr.Textbox(label="Verdict", lines=3),
        gr.Label(label="Probabilités", num_top_classes=2)
    ],
    title       = "🔍 Fake News Detector — BERT",
    description = "Modèle BERT fine-tuné sur 44 898 articles. Entrez un titre et un texte pour analyser sa crédibilité.",
    examples    = examples,
    theme       = gr.themes.Soft(),
    allow_flagging = "never"
)

# share=True génère un lien public accessible depuis n'importe quel navigateur
demo.launch(share=True, debug=False)

# Tu verras un lien comme : https://abc123.gradio.live
# Ce lien fonctionne 72h — copie-le pour ta présentation


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://53304bf07e2194cd2f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
